# 03 — Modèle d'estimation des loyers (Rental Price Model)

Ce notebook filtre les annonces de **location** (TypeOfSale=2) du dataset Immoweb,
entraîne un modèle RandomForest dédié aux loyers et le sauvegarde dans `immo_api/models/rental_model.pkl`.

**Cible :** loyer mensuel en EUR  
**Données :** 20 599 annonces de location (sur 176 910 au total)

In [ ]:
import json, math, warnings, datetime
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

warnings.filterwarnings('ignore')

DATA_PATH   = Path('../data/output.json')
GEO_PATH    = Path('../data/final_dataset.json')
STATBEL_PATH= Path('../data/statbel_features.csv')
MODEL_OUT   = Path('../immo_api/models/rental_model.pkl')
META_OUT    = Path('../immo_api/models/rental_model_metadata.json')

print('Paths OK:', DATA_PATH.exists(), GEO_PATH.exists())

## 1. Load data (incremental JSON decoder)

In [ ]:
decoder = json.JSONDecoder()

def load_json_arrays(path):
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()
    records = []
    idx = 0
    while idx < len(content):
        try:
            obj, end = decoder.raw_decode(content, idx)
            if isinstance(obj, list):
                records.extend(obj)
            idx = end
            while idx < len(content) and content[idx] in ' \t\n\r':
                idx += 1
        except:
            break
    return records

raw = load_json_arrays(DATA_PATH)
print(f'Total records loaded: {len(raw):,}')

# TypeOfSale: 1=sale, 2=rent
from collections import Counter
c = Counter(r.get('TypeOfSale') for r in raw)
print('TypeOfSale distribution:', dict(c))

In [ ]:
# Filter rental records only
rent_raw = [r for r in raw if r.get('TypeOfSale') == 2]
print(f'Rental records: {len(rent_raw):,}')

df = pd.DataFrame(rent_raw)
print(df.shape)
print(df[['Price','LivingArea','BedroomCount' if 'BedroomCount' in df.columns else 'Bedrooms']].describe())

## 2. Load geo data (final_dataset.json)

In [ ]:
geo_raw = load_json_arrays(GEO_PATH)
df_geo = pd.DataFrame(geo_raw)
print(f'Geo records: {len(df_geo):,}')
print('Geo columns:', list(df_geo.columns[:10]))

In [ ]:
# Identify merge key — PropertyId or id
id_col_main = 'PropertyId' if 'PropertyId' in df.columns else 'id'
id_col_geo  = 'PropertyId' if 'PropertyId' in df_geo.columns else 'id'
print(f'Merge on: df["{id_col_main}"] <-> df_geo["{id_col_geo}"]')

geo_cols = [id_col_geo, 'Latitude', 'Longitude', 'DistanceToBrussels',
            'Region', 'FloodingZone', 'EPCkWh', 'MunicipalityAvgPricePerM2']
geo_cols = [c for c in geo_cols if c in df_geo.columns]
df_geo_slim = df_geo[geo_cols].drop_duplicates(subset=[id_col_geo])

df = df.merge(df_geo_slim, left_on=id_col_main, right_on=id_col_geo, how='left')
print(f'After geo merge: {df.shape}')

## 3. Cleaning & feature engineering

In [ ]:
# Standardize column names (PascalCase)
rename = {
    'Bedrooms': 'BedroomCount',
    'SurfaceOfGood': 'SurfaceOfPlot',
    'Heating': 'HeatingType',
    'StateOfBuilding': 'StateOfBuilding',
    'TypeOfProperty': 'TypeOfProperty',
}
df = df.rename(columns={k:v for k,v in rename.items() if k in df.columns})

# Price sanity filter: 200 ≤ rent ≤ 10,000 EUR/month
df = df[df['Price'].between(200, 10000)].copy()
print(f'After price filter: {len(df):,} records')
print(f'Rent range: €{df["Price"].min():.0f} – €{df["Price"].max():.0f}, median €{df["Price"].median():.0f}')

In [ ]:
# Living area filter
df = df[df['LivingArea'].between(15, 1000)].copy()

# Encode categoricals
STATE_MAP = {'AS_NEW':0,'GOOD':1,'TO_BE_DONE_UP':2,'TO_RENOVATE':3,'TO_RESTORE':4}
TYPE_MAP  = {1:0, 2:1, 3:2, 4:3}  # TypeOfProperty codes in raw data
REGION_MAP = {'Flemish Region':0,'Flemish region':0,'Walloon Region':1,'Walloon region':1,
               'Brussels-Capital Region':2,'Brussels':2,
               'Flanders':0,'Wallonia':1}
HEATING_MAP = {'GAS':0,'FUEL_OIL':1,'ELECTRIC':2,'HEAT_PUMP':3,'PELLET':4,'WOOD':5,'SOLAR':6}

df['StateOfBuilding_Num'] = df['StateOfBuilding'].map(STATE_MAP).fillna(1)
df['TypeOfProperty_Num']  = df['TypeOfProperty'].map(TYPE_MAP).fillna(1)  # default: apartment
df['Region_Num']          = df['Region'].map(REGION_MAP).fillna(0) if 'Region' in df.columns else 0
df['HeatingType_Num']     = df['HeatingType'].map(HEATING_MAP).fillna(0) if 'HeatingType' in df.columns else 0

# Derived features
df['ConstructionYear'] = pd.to_numeric(df.get('ConstructionYear', 2000), errors='coerce').fillna(2000)
df['HouseAge'] = (2025 - df['ConstructionYear']).clip(0, 200)
df['BedroomCount'] = pd.to_numeric(df.get('BedroomCount', 1), errors='coerce').fillna(1).clip(0, 20)
df['RoomCount'] = pd.to_numeric(df.get('RoomCount', df['BedroomCount'] + 2), errors='coerce').fillna(3).clip(1, 50)
df['BedroomRatio'] = (df['BedroomCount'] / df['RoomCount'].clip(lower=1)).clip(0, 1)

# Boolean amenities
for col in ['Furnished','Terrace','Garden','SwimmingPool','Fireplace','Lift','Kitchen','Garage']:
    df[col] = pd.to_numeric(df.get(col, 0), errors='coerce').fillna(0).astype(int)

# Location defaults
df['Latitude']  = pd.to_numeric(df.get('Latitude', 50.85), errors='coerce').fillna(50.85)
df['Longitude'] = pd.to_numeric(df.get('Longitude', 4.35), errors='coerce').fillna(4.35)
df['DistanceToBrussels'] = pd.to_numeric(df.get('DistanceToBrussels', 80), errors='coerce').fillna(80)
df['PostalCode'] = pd.to_numeric(df.get('PostalCode', 1000), errors='coerce').fillna(1000).astype(int)
df['MonthlyCharges'] = pd.to_numeric(df.get('MonthlyCharges', 0), errors='coerce').fillna(0)
df['FloorNumber'] = pd.to_numeric(df.get('FloorNumber', 0), errors='coerce').fillna(0).clip(0,50)
df['EPCkWh'] = pd.to_numeric(df.get('EPCkWh', 200), errors='coerce').fillna(200)

print('Encoding done')

In [ ]:
# Merge Statbel features if available
if STATBEL_PATH.exists():
    df_statbel = pd.read_csv(STATBEL_PATH)
    df_statbel['PostalCode'] = df_statbel['PostalCode'].astype(int)
    df = df.merge(df_statbel, on='PostalCode', how='left')
    df['MedianIncome']      = df['MedianIncome'].fillna(df['MedianIncome'].median())
    df['PopulationDensity'] = df['PopulationDensity'].fillna(df['PopulationDensity'].median())
    print('Statbel features merged')
else:
    df['MedianIncome']      = 25000
    df['PopulationDensity'] = 400
    print('Statbel not found, using defaults')

## 4. Feature selection & training

In [ ]:
FEATURE_COLS = [
    'LivingArea', 'BedroomCount', 'RoomCount', 'NumberOfFacades',
    'StateOfBuilding_Num', 'TypeOfProperty_Num', 'Region_Num', 'HeatingType_Num',
    'Furnished', 'Terrace', 'Garden', 'SwimmingPool', 'Fireplace', 'Lift',
    'Kitchen', 'Garage', 'MonthlyCharges',
    'HouseAge', 'BedroomRatio',
    'Latitude', 'Longitude', 'DistanceToBrussels', 'PostalCode',
    'FloorNumber', 'EPCkWh',
    'MedianIncome', 'PopulationDensity',
]

# Keep only available columns
feature_cols = [c for c in FEATURE_COLS if c in df.columns]
print(f'Features used: {len(feature_cols)}')
print(feature_cols)

# Fill any remaining NaN
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

X = df[feature_cols]
y = df['Price']

print(f'\nDataset: {len(X):,} samples, {len(feature_cols)} features')
print(f'Target (rent EUR/month): min={y.min():.0f}, median={y.median():.0f}, max={y.max():.0f}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=4,
    n_jobs=-1,
    random_state=42,
)
model.fit(X_train, y_train)
print('Training complete')

In [ ]:
y_pred = model.predict(X_test)
r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f'R²  = {r2:.4f}')
print(f'MAE = €{mae:.0f}/month')

# Feature importance
fi = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print('\nTop 10 features:')
print(fi.head(10))

## 5. Save model

In [ ]:
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_OUT)
print(f'Model saved → {MODEL_OUT}')

metadata = {
    'model_name':    'RandomForest (Rental)',
    'version':       '1.0',
    'target':        'monthly_rent_eur',
    'r2':            round(r2, 4),
    'mae':           round(mae, 0),
    'dataset_size':  len(X),
    'feature_count': len(feature_cols),
    'features':      feature_cols,
    'trained_at':    datetime.datetime.now().isoformat(timespec='seconds'),
    'rent_range_eur': {'min': int(y.min()), 'median': int(y.median()), 'max': int(y.max())},
}
import json
with open(META_OUT, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Metadata saved → {META_OUT}')
print(json.dumps(metadata, indent=2))

## 6. Quick sanity check

Test a typical Brussels apartment vs Ghent apartment.

In [ ]:
import sys
sys.path.insert(0, str(Path('../immo_api')))
from predictor import predict_rent

cases = [
    {'label': 'Brussels 2BR apt',
     'inp': {'living_area':75,'bedroom_count':2,'room_count':4,'number_of_facades':1,
             'postal_code':1000,'region':'Brussels','furnished':0,'floor_number':2}},
    {'label': 'Gent 3BR house',
     'inp': {'living_area':120,'bedroom_count':3,'room_count':5,'number_of_facades':2,
             'postal_code':9000,'region':'Flanders','garden':1,'terrace':1}},
    {'label': 'Liège 1BR studio',
     'inp': {'living_area':45,'bedroom_count':1,'room_count':2,'number_of_facades':1,
             'postal_code':4000,'region':'Wallonia'}},
]

for c in cases:
    rent = predict_rent(c['inp'])
    print(f"{c['label']:30s}  →  €{rent:,.0f}/month")